# Double Texting

Seamless handling of [double texting](https://docs.langchain.com/langsmith/double-texting) is important for handling real-world usage scenarios, especially in chat applications.

Users can send multiple messages in a row before the prior run(s) complete, and we want to ensure that we handle this gracefully.

## Reject

A simple approach is to [reject](https://docs.langchain.com/langsmith/reject-concurrent) any new runs until the current run completes.

In [1]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

In [3]:
import httpx
from langchain_core.messages import HumanMessage

# Create a thread
thread = await client.threads.create()

# Create to dos
user_input_1 = "Add a ToDo to follow-up with DI Repairs."
user_input_2 = "Add a ToDo to mount dresser to the wall."
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)
try:
    await client.runs.create(
        thread["thread_id"],
        graph_name,
        input={"messages": [HumanMessage(content=user_input_2)]}, 
        config=config,
        multitask_strategy="reject",
    )
except httpx.HTTPStatusError as e:
    print("Failed to start concurrent run", e)

Failed to start concurrent run Thread is already running a task. Wait for it to finish or choose a different multitask strategy.


In [4]:
from langchain_core.messages import convert_to_messages

# Wait until the original run completes
await client.runs.join(thread["thread_id"], run["run_id"])

# Get the state of the thread
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

================================ Human Message =================================

Add a ToDo to follow-up with DI Repairs.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_41491)
 Call ID: call_41491
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'solutions': [], 'status': 'not started', 'time_to_complete': 15, 'task': 'Follow-up with DI Repairs', 'deadline': None}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have added "Follow-up with DI Repairs" to your ToDo list!', 'extras': {'signature': 'El4KXAFpFH0TWE7qYK3K2MZB8iu0Sy5YgL5WxDHQDIW4XOtNAMl76QDVaLjAlMg80fVukA8lMZWvIoLGzuykArZsAQMcpgqpngmGY/1qu3e3kIHartuAKAN/d70RPHyy'}}]


## Enqueue

We can use [enqueue](https://docs.langchain.com/langsmith/enqueue-concurrent) any new runs until the current run completes.

In [5]:
# Create a new thread
thread = await client.threads.create()

# Create new ToDos
user_input_1 = "Send Erik his t-shirt gift this weekend."
user_input_2 = "Get cash and pay nanny for 2 weeks. Do this by Friday."
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

first_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="enqueue",
)

# Wait until the second run completes
await client.runs.join(thread["thread_id"], second_run["run_id"])

# Get the state of the thread
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

================================ Human Message =================================

Send Erik his t-shirt gift this weekend.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_85333)
 Call ID: call_85333
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'time_to_complete': 30, 'deadline': 'this weekend', 'status': 'not started', 'solutions': ['Buy/package t-shirt', 'Ship or hand-deliver to Erik'], 'task': 'Send Erik his t-shirt gift'}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list with the task to send Erik his t-shirt gift this weekend.', 'extras': {'signature': 'El4KXAFpFH0TvTZt9xor3T2zzwW0t+ilzzZvFDORk+Sdz5RVJ+YaE3xV7kd6rdXjMfuzh+EnnJOYm2V+74WZdGpRfglNweSbImnwxsXI0x6cx1IzVuFm6q5KUy/YFSJd'}}]
================================ Human Message ======

## Interrupt

We can use [interrupt](https://docs.langchain.com/langsmith/interrupt-concurrent) to interrupt the current run, but save all the work that has been done so far up to that point.


In [6]:
import asyncio

# Create a new thread
thread = await client.threads.create()

# Create new ToDos
user_input_1 = "Give me a summary of my ToDos due tomrrow."
user_input_2 = "Never mind, create a ToDo to Order Ham for Thanksgiving by next Friday."
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

interrupted_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

# Wait for some of run 1 to complete so that we can see it in the thread 
await asyncio.sleep(1)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="interrupt",
)

# Wait until the second run completes
await client.runs.join(thread["thread_id"], second_run["run_id"])

# Get the state of the thread
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

================================ Human Message =================================

Give me a summary of my ToDos due tomrrow.
================================ Human Message =================================

Never mind, create a ToDo to Order Ham for Thanksgiving by next Friday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_57933)
 Call ID: call_57933
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'time_to_complete': 20, 'task': 'Order Ham for Thanksgiving', 'status': 'not started', 'solutions': ['Order ham online or call local butcher/grocery store'], 'deadline': '2026-10-02T23:59:59Z'}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list to include "Order Ham for Thanksgiving" due by next Friday, October 2, 2026.', 'extras': {'signature': 'El

We can see the initial run is saved, and has status `interrupted`.

In [7]:
# Confirm that the first run was interrupted
print((await client.runs.get(thread["thread_id"], interrupted_run["run_id"]))["status"])

interrupted


## Rollback

We can use [rollback](https://docs.langchain.com/langsmith/rollback-concurrent) to interrupt the prior run of the graph, delete it, and start a new run with the double-texted input.


In [8]:
# Create a new thread
thread = await client.threads.create()

# Create new ToDos
user_input_1 = "Add a ToDo to call to make appointment at Yoga."
user_input_2 = "Actually, add a ToDo to drop by Yoga in person on Sunday."
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

rolled_back_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="rollback",
)

# Wait until the second run completes
await client.runs.join(thread["thread_id"], second_run["run_id"])

# Get the state of the thread
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

================================ Human Message =================================

Actually, add a ToDo to drop by Yoga in person on Sunday.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_97817)
 Call ID: call_97817
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'status': 'not started', 'deadline': '2026-09-27T23:59:59Z', 'time_to_complete': 30, 'task': 'Drop by Yoga in person', 'solutions': ['Visit Yoga studio in person']}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have updated your ToDo list to include dropping by Yoga in person on Sunday.', 'extras': {'signature': 'El4KXAFpFH0Tq+he6Ie7qzHVZafCpkZTAG27/G3LChkaCskOGD6j3e6eAJgwM9nJ5eu6bpMrFtDm5VC0HTQLqlc7abqGVP1H6WrArSLYsgFgrVaq1Kd0HFkN/1YmlflL'}}]


The initial run was deleted.

In [9]:
# Confirm that the original run was deleted
try:
    await client.runs.get(thread["thread_id"], rolled_back_run["run_id"])
except httpx.HTTPStatusError as _:
    print("Original run was correctly deleted")

Original run was correctly deleted


### Summary 

We can see [all the methods summarized](https://docs.langchain.com/langsmith/double-texting):

![Screenshot 2024-11-15 at 12.13.18 PM.png](attachment:ff0af98b-71b1-497a-9c0e-b3519662fd2c.png)